# NewsBot 2.0 — Quickstart

NewsBot Intelligence System 2.0 — ITAI 2373. Run from a cloned project root. Model downloads require internet. Results are measured only when cells actually execute.

## 1. Clone, install and fetch BBC data

In [ ]:
from pathlib import Path
import os, sys
# If opened from GitHub in Colab, clone into /content; otherwise use local checkout.
REPO='https://github.com/iamhuongthn/ITAI2373-NewsBot-Final.git'
if 'google.colab' in sys.modules and not Path('/content/ITAI2373-NewsBot-Final/requirements.txt').exists():
    !git clone {REPO} /content/ITAI2373-NewsBot-Final
if Path('/content/ITAI2373-NewsBot-Final/requirements.txt').exists():
    %cd /content/ITAI2373-NewsBot-Final
else:
    print('Run this notebook from the project root, or open it in Google Colab.')
%pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm
!python scripts/setup_data.py

## 2. Validate dataset

In [ ]:
from config.settings import DATASET, MODEL
from src.data_processing.data_validator import load_bbc
from src.engine import NewsBotEngine
import pandas as pd
news=load_bbc(DATASET)
print('Unique BBC articles:',len(news))
display(news.head(3))

## 3. Train or load classifier

In [ ]:
engine=NewsBotEngine(DATASET,MODEL)
print('Model:',engine.classifier.predict('Technology companies release new software and computer systems.'))

## 4. Topic modeling: LDA and NMF

In [ ]:
from src.analysis.topic_modeler import TopicModeler
for method in ('lda','nmf'):
    tm=TopicModeler(n_topics=7,method=method)
    tm.fit_transform(news.text.tolist())
    print(method.upper(),[(i,tm.get_topic_words(i,7)) for i in range(7)])
    display(tm.visualize_topics())

## 5. Article analysis

In [ ]:
sample=news.iloc[0]['text']
print('Sentiment:',engine.sentiment(sample))
print('Entities:',engine.entities(sample[:1500])[:8])

## 6. Semantic retrieval (downloads model)

In [ ]:
results=engine.search('technology companies internet and software',top_k=3)
for result in results:print(result['index'],round(result['score'],3),result['text'][:120])

## 7. Summarization (downloads model)

In [ ]:
print(engine.summarize(sample[:2200]))

## 8. Multilingual detection and optional translation

In [ ]:
print('Language:',engine.language.detect('Việt Nam đang phát triển công nghệ trí tuệ nhân tạo.'))
# Optional, requires a downloadable translation model:
# print(engine.translate('Technology is changing the world.','en','vi'))

## 9. Conversational search and follow-up

In [ ]:
from src.conversation.interface import ConversationalInterface
bot=ConversationalInterface(engine)
print(bot.ask('technology internet companies'))
print(bot.ask('what is the sentiment?'))

## 10. Automated tests

In [ ]:
!python -m pytest -q